In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_react_agent, AgentExecutor
from langchain.agents.output_parsers import ReActSingleInputOutputParser


# ============================================================
# 2. TOOLS
# ============================================================

@tool
def web_search(query: str) -> str:
    """Search the general web for non-medical information."""
    
    # Replace with actual Web Search API
    return f"Web search results for: {query}"


@tool
def pubmed_search(query: str) -> str:
    """Search PubMed for medical and healthcare information."""
    
    # Replace with actual PubMed API
    return f"PubMed search results for: {query}"


tools = [
    web_search,
    pubmed_search
]


# ============================================================
# 3. AZURE OPENAI
# ============================================================

llm = AzureChatOpenAI(
    azure_endpoint="https://YOUR-RESOURCE.openai.azure.com/",
    api_key="YOUR_API_KEY",
    api_version="2024-10-21",
    azure_deployment="gpt-4.1",
    temperature=0
)


# ============================================================
# 4. PROMPT
# ============================================================

prompt = PromptTemplate.from_template("""
You are an intelligent search agent.

You have two tools:

{tools}

Tool names:
{tool_names}

ROUTING RULE:

- If the question is related to medicine, health,
  disease, symptoms, drugs, treatment, clinical research,
  healthcare or medical science → use pubmed_search.

- For all other questions → use web_search.

Always select the appropriate tool before answering.

Use this format:

Question: the user's question

Thought: decide which tool is appropriate

Action: one of [{tool_names}]

Action Input: the search query

Observation: tool result

Thought: use the result to answer

Final Answer: final response to the user

Question: {input}

{agent_scratchpad}
""")


# ============================================================
# 5. OUTPUT PARSER
# ============================================================

output_parser = ReActSingleInputOutputParser()


# ============================================================
# 6. CREATE REACT AGENT
# ============================================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
    output_parser=output_parser
)


# ============================================================
# 7. AGENT EXECUTOR
# ============================================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)


# ============================================================
# 8. INVOKE AGENT
# ============================================================

response = agent_executor.invoke({
    "input": "What are the symptoms of diabetes?"
})

print(response["output"])